In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame(pd.read_pickle('train_data.pkl'))

In [ ]:
len(df)

In [ ]:
import torch
import numpy as np

X = np.stack(df['confidence'].values)
y = np.stack(df['is_correct'].apply(lambda x: [int(b) for b in x]))

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

In [ ]:
df_val = pd.read_pickle('val_data.pkl')
df_val = pd.DataFrame(df_val)
X_val = np.stack(df_val['confidence'].values)
y_val = np.stack(df_val['is_correct'].apply(lambda x: [int(b) for b in x]))

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)

In [ ]:
import torch.nn as nn

class LogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, input_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(input_dim, input_dim)
        # self.linear3 = nn.Linear(input_dim, input_dim)
        # self.linear4 = nn.Linear(input_dim, input_dim)

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        # x = self.relu(x)
        # x = self.linear3(x)
        # x = self.relu(x)
        # x = self.linear4(x)
        return x

model = LogisticRegression(input_dim=32)

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np

losses = []
val_losses = []

for epoch in range(5000):
    optimizer.zero_grad()
    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()

    current_loss = loss.item()
    losses.append(current_loss)

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss = loss_fn(val_logits, y_val).item()
        val_losses.append(val_loss)

    if epoch % 500 == 0:
        print(f"Epoch {epoch}: loss={loss:.4f}, Val_loss={val_loss:.4f}")


In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(losses, label='Training Loss', color='#0066CC')
plt.plot(val_losses, label='Validation Loss', color='red')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.gca().set_facecolor('#f8f8f8')


plt.savefig('learning_curve.png', dpi=300, bbox_inches='tight')

In [ ]:
# model.load_state_dict(torch.load("/content/layer_1_flan.pth"))
df_test = pd.read_pickle('test_data.pkl')
df_test = pd.DataFrame(df_test)
X_test = np.stack(df_test['confidence'].values)
y_test = np.stack(df_test['is_correct'].apply(lambda x: [int(b) for b in x]))

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
with torch.no_grad():
    logits = model(X_test)
    probs = torch.sigmoid(logits)
    preds = (probs > 0.96).float()
    acc = (preds > y_test).float().mean().item()
    print("Accuracy:", acc)

In [ ]:
torch.save(model.state_dict(), 'latest_filter.pth')